# Selective DNN units
Identify and validate face-, body-, and mixed-selective units in a diverse set of DNNs

In [ ]:
from myutils.models import ModelLoader
from myutils.dataset import ImageDataLoader
from myutils.feature_extractor import FeatureExtractor
from facebody.config import DATA_ROOT
from facebody.selectivity import DNNfloc, compute_sel_dprime
from facebody.plotting import (
    plot_perc_layers, plot_perc_layers_mixed,
    plot_dprime_layers, plot_dprime_dist,
    plot_guided_gradcam,
)

device = "cuda:0"

ALEXNET_LAYERS = [f"conv{i}" for i in range(1, 6)] + [f"fc{i}" for i in range(6, 8)]
VGG16_LAYERS = [f"conv{i}" for i in range(1, 14)] + [f"fc{i}" for i in range(14, 16)]
RESNET50_LAYERS = [f"conv{i}" for i in range(1, 50)]

## Models and layers
- Architectures: AlexNet, VGG16, ResNet50
- Training datasets: Ecoset, ImageNet
- Learning objectives: Supervised, Self-supervised, Untrained

In [ ]:
# Specify models and layers
MODEL_DICT = {
    "alexnet_ecoset": ALEXNET_LAYERS,
    "alexnet_imagenet": ALEXNET_LAYERS,
    "alexnet_untrained": ALEXNET_LAYERS,

    "alexnet-barlow-twins_ecoset": ALEXNET_LAYERS,
    "alexnet-barlow-twins_imagenet": ALEXNET_LAYERS,
    "alexnet-barlow-twins_untrained": ALEXNET_LAYERS,

    "vgg16_ecoset": VGG16_LAYERS,
    "vgg16_imagenet": VGG16_LAYERS,
    "vgg16_untrained": VGG16_LAYERS,

    "resnet50_ecoset": RESNET50_LAYERS,
    "resnet50_imagenet": RESNET50_LAYERS,
    "resnet50_untrained": RESNET50_LAYERS,
}

## Identify selective units
- Functional localizer dataset containing images of faces, bodies, scenes, and objects (288 each), as well as scrambled images (144) (Stigliani et al., 2015)

### Pure and mixed selectivity for faces and bodies

In [ ]:
# Find selective units
floc_loader = ImageDataLoader(
    root=DATA_ROOT / "datasets" / "floc",
    class_names=["face", "body", "scene", "object", "scrambled"],
)

for model, layers in MODEL_DICT.items():
    ml = ModelLoader(model, DATA_ROOT, device)

    fe = FeatureExtractor(ml, layers)
    activs, labels = fe.extract(floc_loader)

    dnn_floc = DNNfloc(
        model, layers,
        class_to_idx=floc_loader.dataset.class_to_idx,
        single_cats=["face", "body"],
        mixed_tuples=[("face", "body")],
    )
    dnn_floc.find_sel_units(activs, labels)

In [ ]:
# Fig. 1B: Plot percentage of selective units across layers
model_dict = {
    "alexnet_ecoset": ALEXNET_LAYERS,
    "alexnet_imagenet": ALEXNET_LAYERS,
    "alexnet-barlow-twins_ecoset": ALEXNET_LAYERS,
    "alexnet-barlow-twins_imagenet": ALEXNET_LAYERS,
    "vgg16_ecoset": VGG16_LAYERS,
    "vgg16_imagenet": VGG16_LAYERS,
    "resnet50_ecoset": RESNET50_LAYERS,
    "resnet50_imagenet": RESNET50_LAYERS,
}

plot_perc_layers(
    model_dict,
    out_path="main/perc_layers.pdf",
)

In [ ]:
# Fig. S1: Plot percentage of selective units across layers in untrained models
model_dict = {
    "alexnet_untrained": ALEXNET_LAYERS,
    "alexnet-barlow-twins_untrained": ALEXNET_LAYERS,
    "vgg16_untrained": VGG16_LAYERS,
    "resnet50_untrained": RESNET50_LAYERS,
}

plot_perc_layers(
    model_dict,
    out_path="suppl/perc_layers_untrained.pdf",
)

### Mixed selectivity for other category combinations

In [ ]:
# Find mixed-selective units for other combinations
floc_loader = ImageDataLoader(
    root=DATA_ROOT / "datasets" / "floc",
    class_names=["face", "body", "scene", "object", "scrambled"],
)

for model, layers in MODEL_DICT.items():
    ml = ModelLoader(model, DATA_ROOT, device)

    fe = FeatureExtractor(ml, layers)
    activs, labels = fe.extract(floc_loader)

    dnn_floc = DNNfloc(
        model, layers,
        class_to_idx=floc_loader.dataset.class_to_idx,
        single_cats=["face", "body", "scene"],
        mixed_tuples=None,
    )
    dnn_floc.find_sel_units(activs, labels)

In [ ]:
# Fig. S2: Plot percentage of mixed-selective units for other combinations across layers
model_dict = {
    "alexnet_ecoset": ALEXNET_LAYERS,
    "alexnet_imagenet": ALEXNET_LAYERS,
    "alexnet-barlow-twins_ecoset": ALEXNET_LAYERS,
    "alexnet-barlow-twins_imagenet": ALEXNET_LAYERS,
    "vgg16_ecoset": VGG16_LAYERS,
    "vgg16_imagenet": VGG16_LAYERS,
    "resnet50_ecoset": RESNET50_LAYERS,
    "resnet50_imagenet": RESNET50_LAYERS,
}

plot_perc_layers_mixed(
    model_dict,
    out_path="suppl/mixed_perc_layers.pdf",
)

## Validate selective units
- Validation dataset containing images of faces, bodies, and objects (150 each; Hemera Photo objects)

In [ ]:
# Confirm selective units
vl = ImageDataLoader(root=DATA_ROOT / "datasets" / "validation")

for model, layers in MODEL_DICT.items():
    ml = ModelLoader(model, DATA_ROOT, device)

    fe = FeatureExtractor(ml, layers)
    activs, labels = fe.extract(vl)

    compute_sel_dprime(
        model, activs, labels, class_to_idx=vl.dataset.class_to_idx,
        img_db="validation", baseline_cat="object",
    )

In [ ]:
# Fig. 2A: Plot face and body selectivity aggregated across layers
model_dict = {
    "alexnet_ecoset": ALEXNET_LAYERS,
    "alexnet_imagenet": ALEXNET_LAYERS,
    "alexnet-barlow-twins_ecoset": ALEXNET_LAYERS,
    "alexnet-barlow-twins_imagenet": ALEXNET_LAYERS,
    "vgg16_ecoset": VGG16_LAYERS,
    "vgg16_imagenet": VGG16_LAYERS,
    "resnet50_ecoset": RESNET50_LAYERS,
    "resnet50_imagenet": RESNET50_LAYERS,
}

plot_dprime_dist(
    model_dict,
    out_path="main/dprime_dist.pdf",
)

In [ ]:
# Fig. S3A: Plot selectivity of units based on validation set across layers
model_dict = {
    "alexnet_ecoset": ALEXNET_LAYERS,
    "alexnet_imagenet": ALEXNET_LAYERS,
    "alexnet-barlow-twins_ecoset": ALEXNET_LAYERS,
    "alexnet-barlow-twins_imagenet": ALEXNET_LAYERS,
    "vgg16_ecoset": VGG16_LAYERS,
    "vgg16_imagenet": VGG16_LAYERS,
    "resnet50_ecoset": RESNET50_LAYERS,
    "resnet50_imagenet": RESNET50_LAYERS,
}

plot_dprime_layers(
    model_dict, img_db="validation", fig_title="Validation",
    out_path="suppl/dprime_layers_val.pdf",
)

In [ ]:
# Fig. S3B: Plot selectivity of units based on validation set across layers in untrained models
model_dict = {
    "alexnet_untrained": ALEXNET_LAYERS,
    "alexnet-barlow-twins_untrained": ALEXNET_LAYERS,
    "vgg16_untrained": VGG16_LAYERS,
    "resnet50_untrained": RESNET50_LAYERS,
}

plot_dprime_layers(
    model_dict, img_db="validation", fig_title="Validation",
    out_path="suppl/dprime_layers_val_untrained.pdf",
)

### Feature visualization
- Visualize feature preferences for each unit type using Guided-GradCAM (Selvaraju et al., 2017)

In [ ]:
# Fig. 2B: Plot feature visualizations for unit types
layer = "conv5"
plot_guided_gradcam(
    model_name="alexnet_ecoset", target_layer=layer, cam_layer=None, controlled=True, device=device,
    img_path=DATA_ROOT / "datasets/guided_gradcam_examples_personal", n_imgs=2, norm_types=True,
    seed=0, fig_title=f"AlexNet-Sup. Ecoset\n{layer}", out_path="main/guided_gradcam.png",
)

In [ ]:
# Fig. S4: Plot feature visualizations for unit types
layer = "conv5"
plot_guided_gradcam(
    model_name="alexnet_ecoset", target_layer=layer, cam_layer=None, controlled=True, device=device,
    img_path=DATA_ROOT / "datasets/guided_gradcam_examples", n_imgs=8, norm_types=True,
    seed=0, fig_title=f"AlexNet-Sup. Ecoset\n{layer}", out_path="suppl/guided_gradcam.pdf",
)